# Imports

In [4]:
import numpy as np
from pathlib import Path
import gymnasium as gym
import numpy as np
import pymatching
import stim
from gymnasium import spaces
from env_raw import QECMaintenanceEnv
import matplotlib.pyplot as plt
import time 
from joblib import Parallel, delayed
from scipy.signal import savgol_filter

# Functions to convert observation space to Q-table index

In [ ]:
def obs_to_bit_flips(obs, bit_flip_indices, num_ancilla):
    # Most recent observations given the history we want
    syndrome_history = obs[:-num_ancilla]
    # Make 2d to have row per timestep  
    rounds = syndrome_history.reshape(-1, num_ancilla)  
    # Most recent syndrome measurement
    latest_round = rounds[-1]  
     # Z ancillas detect bit flips, and we ignore phase flips 
    z_ancillas = latest_round[bit_flip_indices] 
    return z_ancillas

# Given the number of time steps we use for our computation, convert the state to an index
def compute_state(history, num_basic_states):
        # Basic states as a single index, where each state is a digit in base num_basic_states
        state = 0
        for s in history:
            state = state * num_basic_states + s
        return state

def z_index(env):
    # We define the state by concatenating the values of the four Z-ancillas (bit flip measures)
    return [i for i, t in enumerate(env.lattice["ancilla_types"]) if t == "Z"]


# Function for exploring starts 
We initialize the QEC environment with a random error configuration instead of the clean state with no errors in the standard Q-learning

In [ ]:
def exploring_start(env):
    # Reset env, then inject a random error configuration to start from a random state
    obs, info = env.reset()
    
    # Randomly flip each data qubit
    for i in range(env.num_data):
        if np.random.rand() < 0.5:
            # Simulate the logical operations on the data and ancilla qubits, X-error
            env.sim.x(i)         
            # Index 0 for X-error, flip in the error array           
            env.qubit_errors[i, 0] ^= 1     
    
    # Syndrome measurement so the observation shows the injected errors
    env._run_cycle(inject_noise=False)
    # Read the ancilla measurement results
    raw_meas = np.array(env.sim.current_measurement_record()[-env.num_ancilla:], dtype=int)
    # Syndrome change is the XOR with previous measurement to get which ancillas flipped
    diff_syndrome = (raw_meas ^ env.last_syndrome).astype(int)
    # Shift the syndrome history buffer forward by one
    env.syndrome_buffer = np.roll(env.syndrome_buffer, -1, axis=0)
    # Store the new syndrome in the most recent position
    env.syndrome_buffer[-1] = diff_syndrome
    # Update for the next measurement
    env.last_syndrome = raw_meas.copy()
    # Rebuild the observation 
    obs = env._get_obs()
    return obs

# Evaluation episodes

Use the survival time as the reward

In [7]:
def run_evaluations_episode(env, Q_table, num_episodes = 30, history_len=1):
    episode_rewards = []
    # Define the state
    bit_flip_indices = z_index(env)
    # Each of the Z-ancillas can be a 0 or a 1
    num_basic_states = 2 ** len(bit_flip_indices)
    for episode in range(num_episodes):
        # Evaluations also start with a clean state
        obs, _ = env.reset()
        done = truncated = False
        # We keep track of the cumulative reward
        episode_reward = 0
        # Read the ancilla measurements
        bit_flip_measures = obs_to_bit_flips(obs, bit_flip_indices, env.num_ancilla)
        # Convert the binary of errors as strings to an integer to index in the Q-table
        basic_state = int("".join(bit_flip_measures.astype(str)), 2)
        # Fill the history window with the initial state
        history = [basic_state] * history_len
        state = compute_state(history, num_basic_states)
        while not (done or truncated):
            # During evaluation we only use greedy action selection with random action selected from multiple best actions
            max_value = np.max(Q_table[state])
            best_actions = np.where(np.isclose(Q_table[state], max_value))[0]
            env_action = np.random.choice(best_actions)
            # Take one action, we do not use the reward considering the learning implemented in the reward
            # but only the survival point
            obs, _, done, truncated, _ = env.step(env_action)
            episode_reward += 1
            bit_flip_measures = obs_to_bit_flips(obs, bit_flip_indices, env.num_ancilla)
            next_basic_state = int("".join(bit_flip_measures.astype(str)), 2)
            # Update history by dropping the oldest state and appending the new one
            history = history[1:] + [next_basic_state]
            state = compute_state(history, num_basic_states)
        episode_rewards.append(episode_reward)
    return np.mean(episode_rewards)

# Training episodes

In [ ]:
def tabular_rl(env, num_steps=6001, epsilon=0.5, gamma=0.99, alpha=0.1, history_len=1, exploring_starts = False, eval_every=1000):
    # Store the returns after running greedy episodes to evaluate learning
    greedy_returns = []
    # First, we define the state by concatenating the values of the four Z-ancillas (bit flip measures)
    bit_flip_indices = [i for i, t in enumerate(env.lattice["ancilla_types"]) if t == "Z"]
    # Our action space is the possibility to flip each data qubit, or none
    num_actions = env.num_data + 2
    # Given that we have 4 Z-ancillas (for d=3), we have 2^4 = 16 basic states
    num_basic_states = 2 ** len(bit_flip_indices)
    # With history_len observations, we have 16^history_len total states
    num_states = num_basic_states ** history_len
    # Initialize the Q-table
    Q_table = np.zeros((num_states, num_actions))
    visitation_table = np.zeros((num_states, num_actions), dtype=np.int64)
    total_steps = 0

    while total_steps < num_steps:
        if exploring_starts:
            obs, _ = exploring_start(env)
        else:
            obs, _ = env.reset()

        done = truncated = False
        episode_reward = 0
        bit_flip_measures = obs_to_bit_flips(obs, bit_flip_indices, env.num_ancilla)
        
        # Convert the binary of errors as strings to an integer to index in the Q-table
        basic_state = int("".join(bit_flip_measures.astype(str)), 2)
        
        # Fill the history window with the initial state
        history = [basic_state] * history_len
        state = compute_state(history, num_basic_states)
        
        # Random first action for exploring starts
        if exploring_starts:
            env_action = np.random.randint(num_actions)
        elif np.random.rand() < epsilon:
            env_action = np.random.randint(num_actions)
        else:
            max_value = np.max(Q_table[state])
            best_actions = np.where(np.isclose(Q_table[state], max_value))[0]
            env_action = np.random.choice(best_actions)

        # First step
        obs, reward, done, truncated, _ = env.step(env_action)
        episode_reward += reward
        total_steps += 1
        if total_steps % eval_every == 0:
            greedy_returns.append(run_evaluations_episode(env, Q_table, history_len=history_len))
        while not (done or truncated) and total_steps < num_steps:
            visitation_table[state,env_action] += 1
            # Epsilon-greedy action selection
            if np.random.rand() < epsilon:
                env_action = np.random.randint(num_actions)
            else:
                max_value = np.max(Q_table[state])
                # If multiple actions have an estimate which is the highest possible reward
                best_actions = np.where(np.isclose(Q_table[state], max_value))[0]
                # Select randomly between these actions
                env_action = np.random.choice(best_actions)
            # Take one action
            obs, reward, done, truncated, _ = env.step(env_action)
            episode_reward += reward
            total_steps += 1
            bit_flip_measures = obs_to_bit_flips(obs, bit_flip_indices, env.num_ancilla)
            next_basic_state = int("".join(bit_flip_measures.astype(str)), 2)
            # Update history by dropping the oldest state and appending the new one 
            history = history[1:] + [next_basic_state]
            next_state = compute_state(history, num_basic_states)
            # Q-learning update
            if done and not truncated:
                td_target = reward
            else:
                best_next = np.max(Q_table[next_state])
                td_target = reward + gamma * best_next
            Q_table[state, env_action] += alpha * (td_target - Q_table[state, env_action])
            state = next_state
            # Decrease epsilon over time after starting with pure exploration
            epsilon = max(epsilon * 0.99997, 0.05)
            if total_steps % eval_every == 0:
                greedy_returns.append(run_evaluations_episode(env, Q_table, history_len=history_len))
    return Q_table, greedy_returns, visitation_table

# Random agent for comparison of learning curves

In [9]:
def random_agent(env, num_steps=100000, eval_episodes=30, eval_every=1000):
    # Store the returns after running random episodes to use as a baseline
    greedy_returns = []
    num_actions = env.num_data + 2
    total_steps = 0
    while total_steps < num_steps:
        _, _ = env.reset()
        done = truncated = False
        episode_reward = 0
        while not (done or truncated) and total_steps < num_steps:
            env_action = np.random.randint(num_actions)
            _, reward, done, truncated, _ = env.step(env_action)
            episode_reward += reward
            total_steps += 1
            # Evaluate every eval_every steps to match the Q-learning evaluation frequency
            if total_steps % eval_every == 0:
                eval_rewards = []
                for _ in range(eval_episodes):
                    _, _ = env.reset()
                    done = truncated = False
                    episode_reward = 0
                    while not (done or truncated):
                        env_action = np.random.randint(num_actions)
                        _, reward, done, truncated, _ = env.step(env_action)
                        episode_reward += reward
                    eval_rewards.append(episode_reward)
                greedy_returns.append(np.mean(eval_rewards))
    return greedy_returns

# Experiment function

In [ ]:
def single_run(num_steps=100, eval_every=1000):
    # Initialize the environment for each of the agents
    qec_env = QECMaintenanceEnv(render_mode=None)
    qec_env_h = QECMaintenanceEnv(render_mode=None)
    qec_env_rand = QECMaintenanceEnv(render_mode=None)
    
    bit_flip_indices = [i for i, t in enumerate(qec_env.lattice["ancilla_types"]) if t == "Z"]
    num_basic_states = 2 ** len(bit_flip_indices)
    print("Q-learning without history")
    Q_table, rewards_no_history, visitation_no_history = tabular_rl(env=qec_env, num_steps=num_steps, history_len=1, eval_every=eval_every)
    print("Q-learning with history of one time step")
    Q_table_h, rewards_history, visitation_history = tabular_rl(env=qec_env_h, num_steps=num_steps, history_len=2, eval_every=eval_every)
    print("Random Agent")
    rewards_random = random_agent(env=qec_env_rand, num_steps=num_steps, eval_every=eval_every)
    action_labels = [f"flip {i}" for i in range(qec_env.num_data)] + ["no-op", "truncate"]

    return rewards_no_history, rewards_history, rewards_random, visitation_no_history, visitation_history


def smooth_curve(values, window=1, polyorder=1):
    return savgol_filter(values, window_length=window, polyorder=polyorder)

def run_experiments(num_runs=15, num_steps=1000001, smoothing_window=2, eval_every=5000, n_jobs=8):
    start = time.perf_counter()
    results = Parallel(n_jobs=n_jobs)(delayed(single_run)(num_steps, eval_every) for _ in range(num_runs))

    rewards_no_history_all = [r[0] for r in results]
    rewards_history_all = [r[1] for r in results]
    rewards_random_all = [r[2] for r in results]
    visitation_no_history_all = [r[3] for r in results]
    visitation_history_all = [r[4] for r in results]
    np.savez("visitation.npz",
         visitation_no_history=visitation_no_history_all,
         visitation_history=visitation_history_all)

    rewards_no_history_avg = np.mean(rewards_no_history_all, axis=0)
    rewards_history_avg = np.mean(rewards_history_all, axis=0)
    rewards_random_avg = np.mean(rewards_random_all, axis=0)

    # Save the results to allow later (re-)plotting
    np.savez(
    "Q-values.npz",
    rewards_no_history_all = rewards_no_history_all,
    rewards_history_all = rewards_history_all,
    rewards_no_history_avg = rewards_no_history_avg,
    rewards_history_avg = rewards_history_avg,
    rewards_random_all = rewards_random_all,
    rewards_random_avg = rewards_random_avg)

    rewards_no_history_smooth = smooth_curve(rewards_no_history_avg, window=smoothing_window)
    rewards_history_smooth = smooth_curve(rewards_history_avg, window=smoothing_window)
    rewards_random_smooth = smooth_curve(rewards_random_avg, window=smoothing_window)

    xpoints = np.arange(len(rewards_history_smooth)) * eval_every
    end = time.perf_counter()
    print(f"Total runtime: {round(end - start, 2)} seconds")

    plt.plot(xpoints, rewards_no_history_smooth, label="No history")
    plt.plot(xpoints, rewards_history_smooth, label="History")
    plt.plot(xpoints, rewards_random_smooth, label="Random agent")
    plt.xlabel("Training steps")
    plt.ylabel("Total reward")
    plt.legend()
    plt.show()
    
run_experiments()


# Plotting Q-tables

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load data from previously run experiments
visitation_tables1 = np.load("visitation_exp.npz", allow_pickle=True)
visitation_tables2 = np.load("visitation.npz", allow_pickle=True)

for key in ["visitation_no_history", "visitation_history"]:
    for tables, title in [(visitation_tables1, "with exploring start"),
                          (visitation_tables2, "without exploring start")]:
        fig, ax = plt.subplots(figsize=(9, 6))
        table = tables[key][0]
        num_data = table.shape[1] - 2
        # Label each of the actions according to the flip it does
        action_labels = [f"flip {i}" for i in range(num_data)] + ["no-op", "truncate"]
        im = ax.imshow(table, aspect="auto", cmap="viridis", vmax=25000, interpolation="nearest")
        cbar = plt.colorbar(im, ax=ax)
        cbar.ax.set_title("Visit count", fontsize=14, pad=8)
        cbar.ax.tick_params(labelsize=14)
        ax.set_xlabel("Action", fontsize=17)
        ax.set_ylabel("State index", fontsize=17)
        label = "no history" if key == "visitation_no_history" else "history"
        ax.set_title(f"Visitation Counts {title} ({label})", fontsize=15)
        ax.set_xticks(range(len(action_labels)))
        ax.set_xticklabels(action_labels, rotation=45, ha="right", fontsize=12)
        ax.tick_params(axis='y', labelsize=14)
        fig.tight_layout()

plt.show()

#  Replotting results

In [ ]:
from scipy.signal import savgol_filter

def smooth_curve(values, window=1, polyorder=1):
    return savgol_filter(values, window_length=window, polyorder=polyorder)

# Smoothing of each curve
smoothing_window = 35

# We load the data from the saved file
data = np.load("q_timesteps_fixed.npz")
rewards_no_history_avg = data["rewards_no_history_avg"]
rewards_history_avg = data["rewards_history_avg"]
rewards_random_avg = data["rewards_random_avg"]

# For the standard deviation 
rewards_no_history_all = data["rewards_no_history_all"]
rewards_history_all = data["rewards_history_all"]
rewards_random_all = data["rewards_random_all"]

# Calculate the standard deviation per curve
rewards_no_history_std = np.std(rewards_no_history_all, axis=0)
rewards_history_std = np.std(rewards_history_all, axis=0)
rewards_random_std = np.std(rewards_random_all, axis=0)

# Smooth the standard deviation 
rewards_no_history_std_smooth = smooth_curve(rewards_no_history_std, window=smoothing_window)
rewards_history_std_smooth = smooth_curve(rewards_history_std, window=smoothing_window)
rewards_random_std_smooth = smooth_curve(rewards_random_std, window=smoothing_window)

# Smooth the curves
rewards_no_history_smooth = smooth_curve(rewards_no_history_avg, window=smoothing_window)
rewards_history_smooth = smooth_curve(rewards_history_avg, window=smoothing_window)
rewards_random_smooth = smooth_curve(rewards_random_avg, window=smoothing_window)


# Plot the results with the standard deviation
xpoints = np.arange(len(rewards_history_smooth)) * 5000

plt.plot(xpoints, rewards_no_history_smooth, label="No history agent")
plt.fill_between(xpoints, rewards_no_history_smooth - rewards_no_history_std_smooth,
                 rewards_no_history_smooth + rewards_no_history_std, alpha=0.1)

plt.plot(xpoints, rewards_history_smooth, label="History agent")
plt.fill_between(xpoints, rewards_history_smooth - rewards_history_std_smooth,
                 rewards_history_smooth + rewards_history_std, alpha=0.1)

plt.plot(xpoints, rewards_random_smooth, label="Random agent")
plt.fill_between(xpoints, rewards_random_smooth - rewards_random_std_smooth,
                 rewards_random_smooth + rewards_random_std, alpha=0.1)


print("Q-learn no history:", round(rewards_no_history_smooth[-1],2))
print("Q-learn history 1:", round(rewards_history_smooth[-1],2))
plt.ylim(bottom=0, top=100)
plt.xlabel("Time steps (in millions)", fontsize=14)
plt.ylabel("Survival length (in time-steps)", fontsize=14)
plt.title("Q-learning without exploring starts", fontsize=16)
plt.legend(prop = { "size": 14 })
plt.show()

In [ ]:

from scipy.signal import savgol_filter

def smooth_curve(values, window=15, polyorder=1):
    return savgol_filter(values, window_length=window, polyorder=polyorder)

# Smoothing of each curve
smoothing_window = 35

data = np.load("q_timesteps_fixed_explore.npz")
rewards_no_history_avg = data["rewards_no_history_avg"]
rewards_history_avg = data["rewards_history_avg"]
rewards_random_avg = data["rewards_random_avg"]

# For the standard deviation 
rewards_no_history_all = data["rewards_no_history_all"]
rewards_history_all = data["rewards_history_all"]
rewards_random_all = data["rewards_random_all"]

# Calculate the standard deviation per curve
rewards_no_history_std = np.std(rewards_no_history_all, axis=0)
rewards_history_std = np.std(rewards_history_all, axis=0)
rewards_random_std = np.std(rewards_random_all, axis=0)

# Smooth the standard deviation 
rewards_no_history_std_smooth = smooth_curve(rewards_no_history_std, window=smoothing_window)
rewards_history_std_smooth = smooth_curve(rewards_history_std, window=smoothing_window)
rewards_random_std_smooth = smooth_curve(rewards_random_std, window=smoothing_window)

# Smooth the curves
rewards_no_history_smooth = smooth_curve(rewards_no_history_avg, window=smoothing_window)
rewards_history_smooth = smooth_curve(rewards_history_avg, window=smoothing_window)
rewards_random_smooth = smooth_curve(rewards_random_avg, window=smoothing_window)


# Plot the results with the standard deviation
xpoints = np.arange(len(rewards_history_smooth)) * 5000

plt.plot(xpoints, rewards_no_history_smooth, label="No history agent")
plt.fill_between(xpoints, rewards_no_history_smooth - rewards_no_history_std_smooth,
                 rewards_no_history_smooth + rewards_no_history_std, alpha=0.1)

plt.plot(xpoints, rewards_history_smooth, label="History agent")
plt.fill_between(xpoints, rewards_history_smooth - rewards_history_std_smooth,
                 rewards_history_smooth + rewards_history_std, alpha=0.1)

plt.plot(xpoints, rewards_random_smooth, label="Random agent")
plt.fill_between(xpoints, rewards_random_smooth - rewards_random_std_smooth,
                 rewards_random_smooth + rewards_random_std, alpha=0.1)


print("Q-learn no history:", round(rewards_no_history_smooth[-1],2))
print("Q-learn history 1:", round(rewards_history_smooth[-1],2))
plt.ylim(bottom=0, top=100)
plt.xlabel("Time steps (in millions)", fontsize=14)
plt.ylabel("Survival length (in time-steps)", fontsize=14)
plt.title("Q-learning with exploring starts", fontsize=16)
plt.legend(prop = { "size": 14 })
plt.show()